In [2]:
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import chi2

# This opens a file picker dialog — just click your CSV file
from tkinter import filedialog
import tkinter as tk

root = tk.Tk()
root.withdraw()
file_path = filedialog.askopenfilename(title="Select your breast_cancer.csv file")

df = pd.read_csv(file_path)
df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,68,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,4,Positive,Positive,24,1,60,Alive
1,50,White,Married,T2,N2,IIIA,Moderately differentiated,2,Regional,35,Positive,Positive,14,5,62,Alive
2,58,White,Divorced,T3,N3,IIIC,Moderately differentiated,2,Regional,63,Positive,Positive,14,7,75,Alive
3,58,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,18,Positive,Positive,2,1,84,Alive
4,47,White,Married,T2,N1,IIB,Poorly differentiated,3,Regional,41,Positive,Positive,3,1,50,Alive


In [4]:
contingency = pd.crosstab(df['Estrogen Status'], df['Status'])
print(contingency)

chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic: {chi2_stat:.2f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("RESULT: Statistically SIGNIFICANT — ER status genuinely affects survival")
else:
    print("RESULT: Not significant — difference could be due to chance")

Status           Alive  Dead
Estrogen Status             
Negative           161   108
Positive          3247   508

Chi-square statistic: 135.16
P-value: 0.000000
RESULT: Statistically SIGNIFICANT — ER status genuinely affects survival


In [5]:
for col in ['6th Stage', 'Progesterone Status', 'differentiate']:
    ct = pd.crosstab(df[col], df['Status'])
    stat, p, dof, exp = chi2_contingency(ct)
    print(f"{col}: p-value = {p:.6f} → {'SIGNIFICANT ✓' if p < 0.05 else 'NOT significant'}")

6th Stage: p-value = 0.000000 → SIGNIFICANT ✓
Progesterone Status: p-value = 0.000000 → SIGNIFICANT ✓
differentiate: p-value = 0.000000 → SIGNIFICANT ✓


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import numpy as np

df2 = df.copy()

for col in ['Estrogen Status', 'Progesterone Status', '6th Stage']:
    df2[col] = LabelEncoder().fit_transform(df2[col].astype(str))

df2['survived'] = (df2['Status'] == 'Alive').astype(int)

features = ['Age', 'Tumor Size', 'Estrogen Status', 'Progesterone Status', '6th Stage']
X = df2[features]
y = df2['survived']

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0],
    'Odds Ratio': np.exp(model.coef_[0]),
    'Impact': ['Helps survival' if c > 0 else 'Hurts survival' for c in model.coef_[0]]
})
print(coef_df.sort_values('Odds Ratio', ascending=False).to_string(index=False))
print(f"\nModel Accuracy: {model.score(X, y)*100:.1f}%")

            Feature  Coefficient  Odds Ratio         Impact
    Estrogen Status     0.860500    2.364342 Helps survival
Progesterone Status     0.628615    1.875012 Helps survival
         Tumor Size    -0.002886    0.997118 Hurts survival
                Age    -0.022655    0.977599 Hurts survival
          6th Stage    -0.468436    0.625981 Hurts survival

Model Accuracy: 85.5%
